<div style="display:flex; align-items:center; gap:18px; text-align:left">

<div>
<p>Ingeniería en Información y Control de Gestión</p>
<p>Facultad de Economía y Negocios</p>
<p>Introducción a Machine Learning</p>
<p>Semana 06: Regresión predictiva — baseline, métricas y regularización</p>
</div>
</div>

# 06 · Regresión predictiva: baseline, métricas y regularización

El flujo de la clase anterior no cambia. Entre un modelo y otro cambia **la última pieza** del
`Pipeline`, el modelo, y Ridge y Lasso suman el escalado que exigen. Con eso alcanza para recorrer
cuatro modelos.

In [1]:
%%capture
!pip install -q pandas numpy pyarrow scikit-learn

In [2]:
from IPython.display import Markdown, display  # muestra frases con las cifras ya calculadas

def num(x, dec=0):
    """miles con punto, decimales con coma y el menos tipográfico, como el resto del material"""
    texto = f'{x:,.{dec}f}'.replace(',', '@').replace('.', ',').replace('@', '.')
    return texto.replace('-', '−')

def dos(x):
    """la versión de dos decimales que usan las tablas"""
    return num(x, 2)

## El encuadre, declarado antes de tocar el archivo

Sin esto no se puede leer ninguna cifra de la actividad, así que va primero:

- La **unidad de observación es el cliente**. Un registro es un cliente, no una factura ni una línea
  de factura.
- El **objetivo es un conteo**: cuántas veces vuelve a comprar cada cliente en la ventana futura.
- El **momento de la predicción** es el corte entre las dos ventanas. Las fuentes permitidas son solo
  lo anterior a él.
- Por eso la **población no se filtra por lo que pasó después**. El cliente que no vuelve **vale
  cero**, no desaparece del archivo. Filtrarlo sería seleccionar por el resultado, que es justo la
  falta que la clase anterior enseñó a detectar.

### Cargar el archivo

Lo leemos con **pandas**, la librería de tablas del curso. Es un archivo por cliente, construido y
versionado, con el corte temporal ya aplicado. Tres de sus
columnas son un RFM —`recencia_dias`, `facturas_base` y `monto_base`—: se crearon agregando por
cliente las compras anteriores al corte, que es ingeniería de variables.

<!-- contrato: accion=fijar-base -->

In [3]:
import os                # revisa si el archivo está en una carpeta local
import pandas as pd      # la librería de tablas: lee el archivo y lo deja como DataFrame

REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

clientes = pd.read_parquet(BASE + 'clientes_ventas.parquet')
print('Filas y columnas:', clientes.shape)
clientes.head()

Filas y columnas: (3314, 9)


,id_cliente,facturas_base,monto_base,productos_base,ticket_medio_base,recencia_dias,antiguedad_dias,mercado,facturas_futuras
0,12346,1,77183.60,1,77183.60,225,225,reino unido,0
1,12347,5,2790.86,82,558.17,29,267,exportacion,2
2,12348,3,1167.24,21,389.08,148,258,exportacion,1
3,12350,1,294.40,16,294.40,210,210,exportacion,0
4,12352,4,641.51,24,160.38,162,196,exportacion,3


In [4]:
ceros = int((clientes['facturas_futuras'] == 0).sum())
display(Markdown(
    f'De los **{num(len(clientes))}** clientes del archivo, **{num(ceros)}** no vuelven a comprar en '
    f'la ventana futura: el **{num(100 * ceros / len(clientes))} %**. Son el valor más frecuente '
    f'del objetivo y no se pueden sacar.'
))

De los **3.314** clientes del archivo, **1.367** no vuelven a comprar en la ventana futura: el **41 %**. Son el valor más frecuente del objetivo y no se pueden sacar.

### La partición, igual que la clase anterior

De aquí en adelante casi todo viene de **scikit-learn** (`sklearn`), la librería de machine
learning del curso. Cada pieza se importa en la celda donde se usa por primera vez.

Tres pedazos y cada uno con su trabajo. **Entrenamiento** ajusta, **validación** decide entre
alternativas, y **prueba** se abre **una sola vez** cuando ya no queda nada por decidir.

Que validación exista es lo que nos va a permitir elegir un parámetro sin hacer trampa. Si lo
eligiéramos mirando prueba, el error que reportamos dejaría de estimar lo que va a pasar con clientes
nuevos: quedaría optimista por construcción.

<!-- contrato: accion=partir-en-tres -->

In [5]:
from sklearn.model_selection import train_test_split  # sortea la partición

COLUMNAS_NUM = ['facturas_base', 'monto_base', 'productos_base',
                'ticket_medio_base', 'recencia_dias', 'antiguedad_dias']
COLUMNAS_CAT = ['mercado']

X = clientes[COLUMNAS_NUM + COLUMNAS_CAT]
y = clientes['facturas_futuras']

# 20 % a prueba, y del resto un 25 % a validación: quedan 60 / 20 / 20
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42)

print('entrenamiento:', len(X_train), '· validación:', len(X_val), '· prueba:', len(X_test))

entrenamiento: 1988 · validación: 663 · prueba: 663


<p align="center"></p>

Las dos líneas de arriba son **dos particiones, no una**. La segunda cae solo sobre el pedazo
izquierdo de la primera, y por eso `test_size=0.25` sobre el 80 % que quedaba deja un 20 % del total.

Prueba no se toca en ninguno de los dos pasos siguientes. Ésa es toda la disciplina.

# Tiempo 1 · Yo lo hago

## El baseline primero, y con la estrategia que corresponde

La clase anterior ya usó un predictor constante para leer sus cifras contra algo. Hoy ese baseline
deja de ser un dato al costado: es el **competidor**.

El flujo es el mismo de la sesión 5: un `Pipeline` con dos pasos con nombre, **`preparar`** y
**`modelo`**. La preparación se escribe una vez acá y la comparten los modelos que no necesitan
escalar. Ridge y Lasso, más abajo, van a tener la suya, con `StandardScaler`.

Y antes de entrenar hay que **declarar la métrica**, no después. Nuestra vara es el **promedio del
error absoluto, en facturas por cliente**.

<!-- contrato: accion=armar-el-flujo -->

In [6]:
from sklearn.compose import ColumnTransformer  # prepara cada grupo de columnas a su manera
from sklearn.preprocessing import OneHotEncoder  # convierte una categórica en columnas de 0 y 1

# las numéricas pasan tal cual; mercado entra en k−1 columnas, con exportación de referencia
preparacion = ColumnTransformer([
    ('num', 'passthrough', COLUMNAS_NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), COLUMNAS_CAT),
])

La métrica declarada tiene una consecuencia que casi nunca se dice: **para un predictor constante, el
mejor depende de la vara**. El promedio minimiza el error al cuadrado y la mediana minimiza el
error absoluto. Como medimos con la absoluta, el baseline que corresponde es la **mediana**.

Vamos a calcular los dos para que se vea que no da lo mismo. El error se mide con
`mean_absolute_error`, que calcula justamente el promedio del error absoluto.

<!-- contrato: accion=el-baseline -->

In [7]:
from sklearn.pipeline import Pipeline            # encadena la preparación y el modelo
from sklearn.dummy import DummyRegressor         # el modelo que predice siempre lo mismo
from sklearn.metrics import mean_absolute_error  # el promedio del error absoluto

piso_promedio = Pipeline([
    ('preparar', preparacion),
    ('modelo', DummyRegressor(strategy='mean')),
])
piso_mediana = Pipeline([
    ('preparar', preparacion),
    ('modelo', DummyRegressor(strategy='median')),
])
piso_promedio.fit(X_train, y_train)
piso_mediana.fit(X_train, y_train)

# mean_absolute_error recibe primero lo observado y después lo predicho, en ese orden
mae_promedio = mean_absolute_error(y_val, piso_promedio.predict(X_val))
mae_mediana = mean_absolute_error(y_val, piso_mediana.predict(X_val))
print('predecir siempre el promedio:', dos(mae_promedio))
print('predecir siempre la mediana: ', dos(mae_mediana))

predecir siempre el promedio: 1,55
predecir siempre la mediana:  1,39


## Ahora cambiamos la última pieza

Elegir el baseline del promedio nos daría un rival debilitado, y todo lo que el modelo le ganara de
más sería mérito prestado. Seguimos con la **mediana**.

El flujo es el mismo. En el paso `modelo`, `DummyRegressor` sale y entra `LinearRegression`, que
es el modelo que vieron en Estadística y Econometría, ahora con otro propósito: **predecir**, no explicar.

<!-- contrato: accion=el-modelo -->

In [8]:
from sklearn.linear_model import LinearRegression  # la regresión lineal, ajustada por OLS

lineal = Pipeline([
    ('preparar', preparacion),
    ('modelo', LinearRegression()),    # la única pieza que cambia
])
lineal.fit(X_train, y_train)
mae_lineal = mean_absolute_error(y_val, lineal.predict(X_val))

display(Markdown(
    f'El baseline de la mediana estaba en **{dos(mae_mediana)}** facturas de error. El modelo lineal '
    f'baja a **{dos(mae_lineal)}**: le gana por **{dos(mae_mediana - mae_lineal)}**. Recién ahora esa '
    f'cifra de error significa algo, porque tiene contra qué leerse.'
))

El baseline de la mediana estaba en **1,39** facturas de error. El modelo lineal baja a **1,11**: le gana por **0,28**. Recién ahora esa cifra de error significa algo, porque tiene contra qué leerse.

### Leer los coeficientes, con una cautela

`coef_` guarda un número por columna del modelo ya ajustado, y `get_feature_names_out` nos dice a
qué columna corresponde cada uno. Los juntamos para poder leerlos, y los ordenamos con `sort_values`
por **tamaño**, sin importar el signo.

**Ojo con ese orden:** el tamaño de un coeficiente depende de la **unidad** de su columna. Uno por libra
se ve chico aunque mueva mucho la predicción, porque los montos son números grandes. Así que el orden
no dice qué columna pesa más: para comparar hay que traducir cada coeficiente a un cambio con sentido,
como hacen las slides.

**La cautela:** un coeficiente describe una **asociación condicional**, no un efecto. Decir «con todo
lo demás igual» está bien; decir «si subimos esta variable, el objetivo sube» no lo está, porque
nadie intervino nada.

In [9]:
nombres = lineal.named_steps['preparar'].get_feature_names_out()
coeficientes = pd.DataFrame({'coeficiente': lineal.named_steps['modelo'].coef_}, index=nombres)
# ordenados por tamaño, sin importar el signo; el tamaño depende de la unidad de cada columna
coeficientes = coeficientes.sort_values('coeficiente', key=abs, ascending=False)
coeficientes

,coeficiente
num__facturas_base,0.460019
cat__mercado_reino unido,-0.123321
num__productos_base,0.007761
num__antiguedad_dias,-0.006639
num__recencia_dias,0.004162
num__ticket_medio_base,-0.000385
num__monto_base,0.000119


In [10]:
principal = coeficientes.index[0].split('__')[-1]     # sin el prefijo que agrega la preparación
valor = coeficientes['coeficiente'].iloc[0]
display(Markdown(
    f'El coeficiente más grande en magnitud es el de **{principal}**, con **{num(valor, 3)}**. Se lee '
    f'así: entre dos clientes que coinciden en todo lo demás, el que tiene una unidad más de esa '
    f'variable se asocia, en promedio, a {num(valor, 3)} facturas más en la ventana futura. Es una '
    f'comparación entre clientes, no el resultado de intervenir sobre uno.'
))

El coeficiente más grande en magnitud es el de **facturas_base**, con **0,460**. Se lee así: entre dos clientes que coinciden en todo lo demás, el que tiene una unidad más de esa variable se asocia, en promedio, a 0,460 facturas más en la ventana futura. Es una comparación entre clientes, no el resultado de intervenir sobre uno.

# Tiempo 2 · Lo hacemos juntos

## Dos métricas en la misma tabla

Hasta acá usamos una sola vara. Ahora le ponemos nombre a las dos que vamos a usar:

- **MAE** (*mean absolute error*), el **promedio del error absoluto**. Es la que veníamos usando.
- **RMSE** (*root mean squared error*), la **raíz del error cuadrático medio**. Eleva cada error al
  cuadrado antes de promediar, así que un error grande pesa mucho más que varios chicos.

`mean_absolute_error` y `root_mean_squared_error` las calculan, y las dos reciben lo mismo y en el
mismo orden: primero lo **observado** y después lo **predicho**. Hasta ahora solo usamos la primera.

<!-- contrato: accion=metricas-con-nombre -->

In [11]:
from sklearn.metrics import root_mean_squared_error  # la raíz del error cuadrático medio

dos_modelos = {'baseline (mediana)': piso_mediana, 'regresión lineal': lineal}

filas = []
for nombre, modelo in dos_modelos.items():
    prediccion = modelo.predict(X_val)
    filas.append({'modelo': nombre,
                  'MAE': mean_absolute_error(y_val, prediccion),
                  'RMSE': root_mean_squared_error(y_val, prediccion)})

tabla = pd.DataFrame(filas)
tabla.round(2)

,modelo,MAE,RMSE
0,baseline (mediana),1.39,2.50
1,regresión lineal,1.11,1.66


**Para discutir:** en las dos filas el RMSE es más alto que el MAE. ¿Por qué? ¿Y la brecha entre las
dos se abre más en el baseline o en el modelo? Pista: el RMSE eleva cada error al cuadrado, y nuestro
objetivo tiene cola larga.

## Ponerle precio a la complejidad

Ridge y Lasso son el mismo modelo lineal con una **penalización** añadida: además de acercarse a los datos,
tienen que mantener chicos sus coeficientes.

- `Ridge` penaliza los coeficientes **al cuadrado**. Los encoge a todos y ninguno sale del modelo.
- `Lasso` penaliza su **valor absoluto**. Encoge y además lleva algunos exactamente a cero, con lo
  cual saca variables.

Las dos **exigen escalar** las variables, y por eso llevan su propia preparación: la misma de
arriba, con `StandardScaler` en las numéricas, que las deja todas en la misma escala. Sin eso, la
penalización castigaría a la variable que quedó con números grandes por su unidad, no por su
importancia.

In [12]:
from sklearn.preprocessing import StandardScaler  # deja cada numérica con promedio 0 y desviación 1

# la misma preparación, con las numéricas estandarizadas
preparacion_escalada = ColumnTransformer([
    ('num', StandardScaler(), COLUMNAS_NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), COLUMNAS_CAT),
])

### Y el alpha se elige sobre validación

> **Antes de correr esto, un aviso.** Este cuaderno parte el archivo en 60/20/20 de una vez, y las
> slides parten 75/25 y después recortan la validación del entrenamiento. Como la partición es otra,
> **el alpha que gane acá puede no ser el de las slides**. Eso no es un error: es exactamente lo que
> dice el cierre de la clase, que una partición al azar tiene su propia suerte. Cuando
> vean la diferencia, esa es la lección, no una contradicción.

`alpha` es el precio de la penalización. Dos cosas antes de moverlo:

1. **Cada modelo tiene su propia escala de alpha.** No son comparables entre sí, así que cada uno va
   con su grilla. Buscarlos en una grilla común es la forma más rápida de concluir algo falso.
2. **La elección se hace sobre validación**, con prueba todavía cerrada. Elegir el que da el mejor
   error de prueba y después reportar ese mismo error es hacerse trampa a uno mismo.

<!-- contrato: accion=elegir-alpha -->

In [13]:
from sklearn.linear_model import Ridge  # regresión lineal que penaliza los coeficientes al cuadrado

ALPHAS_RIDGE = [10.0, 100.0, 1000.0]

filas_ridge = []
alpha_ridge = None
mejor_error = None
for a in ALPHAS_RIDGE:
    candidato = Pipeline([
        ('preparar', preparacion_escalada),
        ('modelo', Ridge(alpha=a)),
    ])
    candidato.fit(X_train, y_train)
    error = mean_absolute_error(y_val, candidato.predict(X_val))
    filas_ridge.append({'alpha': a, 'MAE en validación': error})
    # nos quedamos con el alpha del error más chico
    if mejor_error is None or error < mejor_error:
        alpha_ridge = a
        mejor_error = error

print('alpha elegido para Ridge:', alpha_ridge)

alpha elegido para Ridge: 1000.0


In [14]:
from sklearn.linear_model import Lasso  # regresión lineal que penaliza su valor absoluto

# el mismo recorrido para Lasso, con su propia grilla
ALPHAS_LASSO = [0.01, 0.1, 1.0]

filas_lasso = []
alpha_lasso = None
mejor_error = None
for a in ALPHAS_LASSO:
    candidato = Pipeline([
        ('preparar', preparacion_escalada),
        ('modelo', Lasso(alpha=a)),
    ])
    candidato.fit(X_train, y_train)
    error = mean_absolute_error(y_val, candidato.predict(X_val))
    filas_lasso.append({'alpha': a, 'MAE en validación': error})
    if mejor_error is None or error < mejor_error:
        alpha_lasso = a
        mejor_error = error

print('alpha elegido para Lasso:', alpha_lasso)

alpha elegido para Lasso: 1.0


In [15]:
todas = []
for nombre, filas in [('Ridge', filas_ridge), ('Lasso', filas_lasso)]:
    for fila in filas:
        todas.append({'modelo': nombre,
                      'alpha': fila['alpha'],
                      'MAE en validación': fila['MAE en validación']})

pd.DataFrame(todas).round(4)

,modelo,alpha,MAE en validación
0,Ridge,10.00,1.1074
1,Ridge,100.00,1.0947
2,Ridge,1000.00,1.0879
3,Lasso,0.01,1.1045
4,Lasso,0.10,1.0803
5,Lasso,1.00,1.0464


In [16]:
ridge = Pipeline([
    ('preparar', preparacion_escalada),
    ('modelo', Ridge(alpha=alpha_ridge)),
])
lasso = Pipeline([
    ('preparar', preparacion_escalada),
    ('modelo', Lasso(alpha=alpha_lasso)),
])
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

coef_lasso = pd.DataFrame({'coeficiente': lasso.named_steps['modelo'].coef_}, index=nombres)
en_cero = list(coef_lasso[coef_lasso['coeficiente'].abs() < 1e-10].index)
# no lo afirmamos: lo contamos, igual que el de Lasso
vivos_ridge = int((abs(ridge.named_steps['modelo'].coef_) > 1e-10).sum())
display(Markdown(
    f'Lasso con alpha {alpha_lasso} llevó a cero **{len(en_cero)} de {len(coef_lasso)}** coeficientes: '
    f'{", ".join(en_cero) if en_cero else "ninguno"}. Ridge, en cambio, conserva '
    f'{vivos_ridge} de {len(coef_lasso)}: encogió, pero no sacó a nadie.'
))

Lasso con alpha 1.0 llevó a cero **6 de 7** coeficientes: num__monto_base, num__productos_base, num__ticket_medio_base, num__recencia_dias, num__antiguedad_dias, cat__mercado_reino unido. Ridge, en cambio, conserva 7 de 7: encogió, pero no sacó a nadie.

# Tiempo 3 · Lo hacen ustedes

## La decisión

Tienen cuatro modelos sobre la mesa y hay que **llevar uno a una gerencia**. La actividad es
decidirlo y defenderlo por escrito en **tres líneas**.

La decisión **no es obvia**, y ahí está el punto. Miren la tabla de alpha que calcularon más arriba
antes de responder:

- El MAE más bajo es **un** argumento, y no siempre gana por mucho.
- Un modelo que usa **menos columnas** es más fácil de explicar y de mantener. Ése es un argumento
  distinto —**simplicidad**— y hay que decidir cuánto pesa cuando la diferencia de error es chica.
- Y ojo con esto: **el alpha lo eligió una sola partición de validación**. Con otra partición al azar el
  ganador puede ser otro. Cuánta confianza le dan a esa elección es parte de la decisión: si gana el
  modelo más simple de todos, pregúntense si le creerían igual con clientes nuevos. El cierre lo pone a
  prueba.

**Criterio de éxito:** elegir el baseline que corresponde a la métrica y justificarlo; leer un
coeficiente en unidades del negocio, con su cautela; leer MAE y RMSE y explicar por qué difieren;
nombrar qué le hace la penalización a los coeficientes y en qué se diferencian Ridge y Lasso; y
defender una elección de modelo con la tabla delante.

**Evidencia observable:** la tabla comparativa de los cuatro modelos con sus dos métricas, la lista
de coeficientes que Lasso llevó a cero, una frase que lea un coeficiente, y las tres líneas de la
decisión justificada.

**Una advertencia antes de mirar la tabla.** Las filas de Ridge y Lasso salen un poco optimistas,
porque su alpha se eligió con esta misma validación: la tabla sirve para **decidir**, no para reportar.
La cifra que se reporta llega al final, con la prueba.

In [17]:
# armen la tabla de los cuatro modelos sobre validación y mírenla antes de decidir
cuatro_modelos = {'baseline (mediana)': piso_mediana,
                  'regresión lineal': lineal,
                  f'ridge (alpha={alpha_ridge})': ridge,
                  f'lasso (alpha={alpha_lasso})': lasso}

filas = []
for nombre, modelo in cuatro_modelos.items():
    prediccion = modelo.predict(X_val)
    filas.append({'modelo': nombre,
                  'MAE': mean_absolute_error(y_val, prediccion),
                  'RMSE': root_mean_squared_error(y_val, prediccion)})

comparacion = pd.DataFrame(filas)
# el baseline no usa ningún predictor; los otros, los que no quedaron en cero
comparacion['predictores'] = [0, len(nombres), len(nombres), len(nombres) - len(en_cero)]
comparacion.round(2)

,modelo,MAE,RMSE,predictores
0,baseline (mediana),1.39,2.50,0
1,regresión lineal,1.11,1.66,7
2,ridge (alpha=1000.0),1.09,1.60,7
3,lasso (alpha=1.0),1.05,1.44,1


### Su respuesta

Escriban acá las tres líneas. Qué modelo llevarían, con qué cifra lo sostienen, y qué renuncian al
elegirlo.

1.
2.
3.

Y una frase más: lean **un coeficiente** del modelo que eligieron, con su unidad, con «con todo lo
demás igual», y diciendo por qué no es el efecto de intervenir.

-

# Cierre

## Abrir la prueba, una sola vez

Ya no queda nada por decidir: la métrica está declarada, el baseline elegido y los alpha fijados
sobre validación.

Falta **un paso** antes de medir, y es el que más se olvida. Validación ya hizo su trabajo: eligió el
alpha. Así que ahora se **reajustan los cuatro modelos con todo el entrenamiento** —entrenamiento más
validación— y recién esos son los que se llevan a prueba. Reservar los clientes de validación, que ya no
hacen falta, sería tirar datos.

**Una precaución de scikit-learn.** Un `Pipeline` ajusta el mismo objeto de preparación que recibe:
no hace una copia. Si estos modelos finales usaran `preparacion_escalada`, ajustarlos con todo el
entrenamiento cambiaría también, sin aviso, a los `ridge` y `lasso` de arriba. Por eso las dos
preparaciones se escriben de nuevo.

In [18]:
# preparaciones nuevas, para no tocar las que ya ajustaron los modelos de arriba
preparacion_final = ColumnTransformer([
    ('num', 'passthrough', COLUMNAS_NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), COLUMNAS_CAT),
])
preparacion_escalada_final = ColumnTransformer([
    ('num', StandardScaler(), COLUMNAS_NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), COLUMNAS_CAT),
])

In [19]:
# paso 4 · con los alpha ya elegidos, se reajusta con todo el entrenamiento
finales = {
    'baseline (mediana)': Pipeline([('preparar', preparacion_final),
                                    ('modelo', DummyRegressor(strategy='median'))]),
    'regresión lineal': Pipeline([('preparar', preparacion_final),
                                  ('modelo', LinearRegression())]),
    f'ridge (alpha={alpha_ridge})': Pipeline([('preparar', preparacion_escalada_final),
                                              ('modelo', Ridge(alpha=alpha_ridge))]),
    f'lasso (alpha={alpha_lasso})': Pipeline([('preparar', preparacion_escalada_final),
                                              ('modelo', Lasso(alpha=alpha_lasso))]),
}

for nombre, modelo in finales.items():
    finales[nombre] = modelo.fit(X_trainval, y_trainval)

print('reajustados con', len(X_trainval), 'clientes en vez de', len(X_train))

reajustados con 2651 clientes en vez de 1988


Recién ahora se abre el conjunto de prueba, y se abre una vez.

<!-- contrato: accion=abrir-la-prueba -->

In [20]:
# el mismo bucle de recién: cambian el conjunto y los modelos, que ahora son los reajustados
filas = []
for nombre, modelo in finales.items():
    prediccion = modelo.predict(X_test)
    filas.append({'modelo': nombre,
                  'MAE': mean_absolute_error(y_test, prediccion),
                  'RMSE': root_mean_squared_error(y_test, prediccion)})

prueba = pd.DataFrame(filas)
prueba.round(2)

,modelo,MAE,RMSE
0,baseline (mediana),1.44,2.84
1,regresión lineal,1.14,1.84
2,ridge (alpha=1000.0),1.14,1.77
3,lasso (alpha=1.0),1.16,1.81


## Un límite del modelo, a la vista

Mirar dónde se equivoca el modelo sirve para nombrar su límite, no para arreglarlo hoy. El más visible
está en sus predicciones de prueba.

In [21]:
prediccion = finales['regresión lineal'].predict(X_test)
negativas = int((prediccion < 0).sum())
aviso = (f'llega a predecir **{dos(prediccion.min())}** facturas, y nadie compra un número negativo de '
         f'veces' if negativas else
         f'no llegó a predecir ningún negativo en esta partición, pero su mínimo es '
         f'**{dos(prediccion.min())}** y nada en el modelo lo impide')
display(Markdown(
    f'De los **{num(len(X_test))}** clientes de prueba, el modelo lineal predice '
    f'**{num(negativas)}** valores negativos: {aviso}. Es el límite de haber tratado un **conteo** '
    f'como si fuera una cantidad continua. La familia de modelos que le corresponde existe y queda '
    f'fuera de esta clase.'
))

De los **663** clientes de prueba, el modelo lineal predice **3** valores negativos: llega a predecir **−17,81** facturas, y nadie compra un número negativo de veces. Es el límite de haber tratado un **conteo** como si fuera una cantidad continua. La familia de modelos que le corresponde existe y queda fuera de esta clase.

## La pregunta que queda abierta

Los alpha de hoy salieron de **una** partición de validación, y esa partición tiene su propia
suerte: con otra, el elegido podría ser otro. ¿Cómo se elige un parámetro sin depender de una sola
partición?

Eso es la sesión 8, con validación cruzada y búsqueda sobre grilla.

## Atribución de datos

- **Creador:** Chen, D. (2012)
- **Fuente:** [UCI Machine Learning Repository — *Online Retail II*, dataset 502](https://archive.ics.uci.edu/dataset/502/online+retail+ii)
- **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- **Modificación:** obra derivada en tres pasos. Las columnas originales se renombraron al español en
  `snake_case`; se aplicaron las siete decisiones de limpieza de la clase 4; y
  `_prep_clientes_ventas.py` agregó por cliente con un corte temporal, quedando 3.314 registros y
  9 columnas.